### The universal pattern for an event stream with varying optional fields is:

- Declare one explicit, nullable, superset schema covering every field that can ever appear across all event types. This solves “some fields don’t show up in some files” — by design, not by accident.
- Fix the producer so that when a field is present, its JSON representation actually matches the declared type (your convert_dtypes() fix).
- Keep cloudFiles.schemaEvolutionMode = "rescue" as a safety net, not your primary mechanism — it should catch truly unexpected stuff (a brand-new field Wikimedia adds tomorrow), not routinely absorb fields you already know about.\
`(pure schema inference, no explicit schema) is the wrong default for this kind of source — inference is unstable (can flip types batch-to-batch, which is literally what caused your original timestamp failure) and gives you no durable contract to build Silver/Gold on top of.`

In [0]:
%sql
select 'silver' as source_table,
       min(timestamp) as first_occurrence,
       max(timestamp) as last_occurrence,
       count(*) as total_events
from workspace.silver.wikimedia_silver

union all

select 'bronze' as source_table,
       min(try_cast(timestamp as timestamp)) as first_occurrence,
       max(try_cast(timestamp as timestamp)) as last_occurrence,
       count(*) as total_events
from workspace.bronze.wikimedia_bronze;

In [0]:
%run ./params_config

In [0]:
df=dbutils.fs.ls(SOURCE_PATH)
len(df)

In [0]:

df = spark.read.json(SOURCE_PATH)






In [0]:
df=spark.read.table(f"{catalog}.{bronze_schema}.{bronze_table}")
df.printSchema()


In [0]:
%sql
drop table if exists workspace.bronze.wikimedia_bronze

In [0]:

dbutils.fs.rm(schema_location_bronze, recurse=True)
dbutils.fs.mkdirs(schema_location_bronze)
dbutils.fs.rm(checkpoint_bronze, recurse=True)
dbutils.fs.mkdirs(checkpoint_bronze)


In [0]:
spark.sql(f"select count(*) from {catalog}.{bronze_schema}.{bronze_table}
          union all 
          select count(*) from {catalog}.{silver_schema}.{silver_table}")

In [0]:
%sql
SELECT *
FROM workspace.silver.wikimedia_silver
WHERE CAST(ingested_at AS DATE) = current_date()
order by ingested_at desc